# 01 - Dataset statistics (brief requirement)

 This notebook computes and saves:images per split, image resolution, number of classes, and the class-pixel distribution (to show imbalance).
- `results/dataset_stats.json` - the numbers,
- `results/figures/class_distribution.png` - the imbalance chart for the report,
- `results/class_weights.json` - median-frequency weights used later by the `weighted_ce` loss.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys, os
import numpy as np
sys.path.insert(0, '/content/drive/MyDrive/coco_seg/project/code')
import config as C
import data, engine, losses, models, agent_search, report_utils
C.ensure_dirs()
print('GPU:', __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'NONE - Runtime > Change runtime type > T4 GPU')

GPU: NONE - Runtime > Change runtime type > T4 GPU


### Counts and resolution

In [3]:
frames = data.load_frames()
from PIL import Image
import numpy as np, random
sizes = []
for f in frames['train'].sample(60, random_state=42).itertuples():
    with Image.open(f.image) as im:
        sizes.append(im.size)
w = [s[0] for s in sizes]; h = [s[1] for s in sizes]
stats = {
    'images_per_split': {s: len(f) for s, f in frames.items()},
    'num_classes': C.NUM_CLASSES,
    'class_names': C.CLASS_NAMES,
    'resolution': {'width_mode': int(np.median(w)), 'height_mode': int(np.median(h)),
                   'width_min': min(w), 'width_max': max(w),
                   'height_min': min(h), 'height_max': max(h)},
}
print(stats)

masks index: 10000 rows | splits: {'train': 8000, 'val': 1000, 'test': 1000} | missing files: 0
                                                                            image                                                          mask  split
0  /content/drive/MyDrive/coco_seg/fiftyone/coco-2017/train/data/000000307967.jpg  /content/drive/MyDrive/coco_seg/masks/train/000000307967.png  train
1  /content/drive/MyDrive/coco_seg/fiftyone/coco-2017/train/data/000000154345.jpg  /content/drive/MyDrive/coco_seg/masks/train/000000154345.png  train
{'images_per_split': {'train': 8000, 'val': 1000, 'test': 1000}, 'num_classes': 13, 'class_names': ['background', 'person', 'car', 'dog', 'cat', 'chair', 'bottle', 'bus', 'bicycle', 'horse', 'cup', 'laptop', 'bird'], 'resolution': {'width_mode': 640, 'height_mode': 480, 'width_min': 398, 'width_max': 640, 'height_min': 224, 'height_max': 640}}


### Class-pixel distribution (the imbalance chart)

In [4]:
SAMPLE = 400   # masks sampled to estimate the distribution
counts = np.zeros(C.NUM_CLASSES, dtype=np.int64)
sample = frames['train'].sample(min(SAMPLE, len(frames['train'])), random_state=42)
for f in sample.itertuples():
    with Image.open(f.mask) as m:
        counts += np.bincount(np.asarray(m).ravel(), minlength=C.NUM_CLASSES)
share = counts / counts.sum()
for name, c, s in zip(C.CLASS_NAMES, counts, share):
    print(f'{name:12s} pixels={c:>12,}  share={s:.4%}')
stats['class_pixel_counts_sampled'] = counts.tolist()
stats['class_pixel_share'] = share.tolist()
stats['distribution_sample_size'] = int(len(sample))

from data_viz import bar_chart
import os
bar_chart(C.CLASS_NAMES, share, os.path.join(C.FIGURES_DIR, 'class_distribution.png'),
          title='Class-pixel distribution (train sample)', ylabel='share of pixels')
report_utils.save_result('dataset_stats', stats)

weights = losses.median_frequency_weights(counts, C.NUM_CLASSES)
report_utils.save_result('class_weights', {'class_names': C.CLASS_NAMES, 'median_frequency_weights': weights})
print('saved dataset_stats.json, class_distribution.png, class_weights.json')

background   pixels=  90,221,433  share=81.6335%
person       pixels=  13,346,888  share=12.0764%
car          pixels=     696,009  share=0.6298%
dog          pixels=     720,337  share=0.6518%
cat          pixels=     907,972  share=0.8215%
chair        pixels=     945,347  share=0.8554%
bottle       pixels=     254,532  share=0.2303%
bus          pixels=   1,417,823  share=1.2829%
bicycle      pixels=     218,618  share=0.1978%
horse        pixels=     424,391  share=0.3840%
cup          pixels=     291,646  share=0.2639%
laptop       pixels=     664,992  share=0.6017%
bird         pixels=     410,087  share=0.3711%
saved dataset_stats.json, class_distribution.png, class_weights.json


notice: the chart drops fast - background and person dominate, classes like cup and laptop are rare. That imbalance is why `weighted_ce` and the composite loss exist; mention this in your report's dataset section.